<a href="https://colab.research.google.com/github/prasath25/Hands-on/blob/main/Experiment_3_Prompt_Engineering_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 3 — Prompt Engineering & Advanced Text Generation with OpenAI

**Goal:** Help students see that output quality depends heavily on how the task is described.

### Learning outcomes
By the end of this notebook, students can:
1. Call the OpenAI API from Python.
2. Compare weak and engineered prompts.
3. Apply zero-shot and few-shot prompting.
4. Request machine-readable JSON.
5. Use constraints, roles, examples, and evaluation prompts.
6. Build a small prompt-engineering challenge.

### Classroom flow
**Predict → Run → Inspect → Modify → Challenge**

## 0. Setup

Create an OpenAI API key and keep it private.

- In Colab, you can paste it when prompted.
- In Jupyter, the same prompt works.
- Never place the key directly inside code that will be shared.

In [ ]:
!pip -q install -U openai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 48.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [ ]:
import os
import getpass
import json
import pandas as pd
from openai import OpenAI
os.environ.pop("OPENAI_API_KEY", None)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# A broadly available model for teaching.
# Change this if your OpenAI project has access to a different model.
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

print("Using model:", MODEL)

Enter OPENAI_API_KEY: ··········
Using model: gpt-4.1-mini


## 1. First API call

Before teaching prompt engineering, verify that the API works.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Reply with exactly: OpenAI API is working."
)
print(response.output_text)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

## 2. Helper function

This keeps later cells easy to read.

In [ ]:
def ask_openai(prompt, model=MODEL):
    response = client.responses.create(
        model=model,
        input=prompt
    )
    return response.output_text

## 3. Zero-shot classification

**Ask students first:**  
If we do not provide examples, how will the model know the labels we expect?

In [ ]:
messages = [
    "My Wi-Fi disconnects every five minutes.",
    "Why was I charged twice this month?",
    "My parcel was expected yesterday but it has not arrived."
]

for text in messages:
    prompt = f'''
Classify the customer message into exactly one category:
Technical, Billing, Delivery, or Other.

Customer message:
{text}

Return only the category.
'''
    print(text, "->", ask_openai(prompt))

### Discussion
This is **zero-shot prompting** because the model receives instructions but no worked examples.

## 4. Weak prompt vs engineered prompt

Ask students to predict which prompt will be more consistent.

In [ ]:
customer_text = '''
I ordered headphones three days ago. The tracking page still says processing.
I already contacted support once and I need the item before Friday.
'''

weak_prompt = f"Help with this customer message: {customer_text}"

engineered_prompt = f'''
You are a customer-support analysis assistant.

Analyze the message below.

Tasks:
1. Identify the main intent.
2. Identify the customer's emotion.
3. State the likely urgency as Low, Medium, or High.
4. Recommend one next action for the support team.
5. Do not invent facts not present in the message.

Customer message:
{customer_text}

Return the answer using these headings:
Intent:
Emotion:
Urgency:
Recommended action:
'''

print("----- WEAK PROMPT -----")
print(ask_openai(weak_prompt))

print("\n----- ENGINEERED PROMPT -----")
print(ask_openai(engineered_prompt))

### Teaching point
A useful prompt commonly includes:

**Role + Task + Context + Constraints + Output Format**

The model is not reading your mind. The prompt is part of the program.

## 5. Few-shot prompting

Now we teach the model the expected pattern using examples.

In [ ]:
few_shot_prompt = '''
Classify each customer message into one of these categories:
Technical, Billing, Delivery.

Examples:
Message: "My router keeps restarting."
Category: Technical

Message: "The invoice amount is incorrect."
Category: Billing

Message: "The courier has not delivered my package."
Category: Delivery

Now classify:
Message: "I paid already, but the portal still shows an outstanding balance."

Return only the category.
'''

print(ask_openai(few_shot_prompt))

### Experiment
Remove one example. Change an example. Add a confusing example.

Observe whether the model output changes.

## 6. Request JSON output

For applications, predictable structure is often more useful than prose.

This example asks the model to return JSON and validates it in Python.

In [ ]:
json_prompt = f'''
Analyze the customer message below.

Return ONLY valid JSON with exactly these keys:
intent, emotion, urgency, recommended_action

Rules:
- urgency must be one of: Low, Medium, High
- do not include Markdown
- do not invent missing facts

Customer message:
{customer_text}
'''

raw = ask_openai(json_prompt)
print("Raw model response:")
print(raw)

try:
    parsed = json.loads(raw)
    print("\nParsed Python object:")
    print(parsed)
except json.JSONDecodeError:
    print("\nThe response was not valid JSON. This is a useful classroom failure case.")

## 7. Same information, different instructions

This demonstrates that the **instruction changes the output**, even when the source text is identical.

In [ ]:
source = '''
A university library is open from 8 AM to 8 PM on weekdays.
Students may borrow five books for fourteen days.
Late returns may temporarily block new borrowing.
'''

prompts = {
    "summary": f"Summarize this in one sentence:\n{source}",
    "student_faq": f"Turn this into a 3-question student FAQ:\n{source}",
    "quiz": f"Create 3 multiple-choice questions from this text. Include answers.\n{source}",
    "strict_extraction": f"Extract only opening hours, borrowing limit, and loan duration.\n{source}",
}

for name, prompt in prompts.items():
    print(f"\n### {name.upper()} ###")
    print(ask_openai(prompt))

## 8. Constraint experiment

Prompt A allows free-form writing. Prompt B limits length and style.

Ask students which answer would be easier to display in a mobile app.

In [ ]:
topic = "Explain embeddings to a first-year computer science student."

prompt_a = topic

prompt_b = f'''
{topic}

Constraints:
- maximum 70 words
- use one analogy
- avoid mathematical notation
- end with one practical example
'''

print("A:\n", ask_openai(prompt_a))
print("\nB:\n", ask_openai(prompt_b))

## 9. Prompt evaluation

Instead of only generating an answer, use a second prompt to critique the first answer.

In [ ]:
task = '''
Explain retrieval-augmented generation to a beginner in under 100 words.
Mention retrieval, context, and generation.
'''

answer = ask_openai(task)

evaluation_prompt = f'''
Evaluate the answer below against these criteria:
1. Beginner-friendly
2. Mentions retrieval
3. Mentions context
4. Mentions generation
5. Under 100 words

Answer:
{answer}

Return:
- PASS or NEEDS IMPROVEMENT
- one short reason
'''

print("ANSWER:\n", answer)
print("\nEVALUATION:\n", ask_openai(evaluation_prompt))

# Student Challenge

Given this complaint:

> "My phone was supposed to arrive on Monday. Tracking still shows processing. I contacted support yesterday and was told to wait. I need it for a trip this weekend."

Create a prompt that extracts:

- intent
- emotion
- product
- expected delivery
- current status
- previous contact
- requested/likely next action

### Extension
1. Ask for JSON.
2. Validate the JSON in Python.
3. Change one constraint and compare the output.

# Key takeaway

Prompt engineering is not about discovering a single “magic prompt.”

It is systematic task design:

**Task → Context → Examples → Constraints → Output Format → Evaluation**